In [1]:
import os
import sys
from pathlib import Path

# Get the notebook's directory
NOTEBOOK_DIR = Path(os.getcwd())
BACKEND_DIR = NOTEBOOK_DIR.parent.parent / "Skin_Lesion_Classification_backend"
ML_DIR = BACKEND_DIR / "ml"

# Add backend ml/ to sys.path so imports like `from src.data.dataset` work
if str(ML_DIR) not in sys.path:
    sys.path.insert(0, str(ML_DIR))

print(f"Notebook directory: {NOTEBOOK_DIR}")
print(f"Backend directory:  {BACKEND_DIR}")
print(f"ML directory:      {ML_DIR}")
print(f"Python:            {sys.executable}")

# Verify backend exists
if not BACKEND_DIR.exists():
    print(f"\u274c ERROR: Backend not found at {BACKEND_DIR}")
    print("Make sure the backend repo is cloned at the same level as the frontend repo.")
else:
    print("\u2705 Backend directory found.")

Notebook directory: c:\Users\saiyu\Desktop\projects\KI_projects\Skin_Lesion_GRADCAM_Classification\Skin_Lesion_Classification_frontend\notebooks
Backend directory:  c:\Users\saiyu\Desktop\projects\KI_projects\Skin_Lesion_GRADCAM_Classification\Skin_Lesion_Classification_backend
ML directory:      c:\Users\saiyu\Desktop\projects\KI_projects\Skin_Lesion_GRADCAM_Classification\Skin_Lesion_Classification_backend\ml
Python:            c:\Users\saiyu\Desktop\projects\KI_projects\Skin_Lesion_GRADCAM_Classification\.venv\Scripts\python.exe
✅ Backend directory found.


# RQ4 — Does Inter-method Disagreement Predict Misclassification?
## XAI Skin Lesion Classification — Research Question 4

**Hypothesis**: When different CAM methods disagree on *which regions* are important,
the model is more likely to be uncertain or wrong.

**Intuition**: If GradCAM and GradCAM++ both highlight the same lesion border,
the model has a "consistent story". If they highlight completely different regions,
the model might be confused — which is a signal of uncertainty.

---

## CONCEPT: Jaccard Similarity

If GradCAM highlights pixels A and GradCAM++ highlights pixels B:
  Jaccard = |A ∩ B| / |A ∪ B|

Jaccard = 1.0 → perfect agreement (same pixels highlighted)
Jaccard = 0.0 → complete disagreement (no overlap at all)

Why Jaccard and not correlation?
Because CAM maps are sparse binary regions after thresholding.
Correlation would be dominated by the background zeros.

---

## CELL 1: Define Agreement Metrics

In [ ]:
import numpy as np
from scipy.stats import spearmanr

def jaccard_similarity(cam1, cam2, threshold=0.5):
    """Measure overlap between two thresholded CAM maps."""
    b1 = (cam1 >= threshold).astype(bool)
    b2 = (cam2 >= threshold).astype(bool)
    intersection = np.logical_and(b1, b2).sum()
    union = np.logical_or(b1, b2).sum()
    return float(intersection / union) if union > 0 else 0.0

def spearman_agreement(cam1, cam2):
    """
    Rank correlation between raw CAM values (no threshold needed).
    More sensitive than Jaccard — picks up partial agreement.
    """
    r, _ = spearmanr(cam1.flatten(), cam2.flatten())
    return r

# Test with dummy maps
cam_a = np.zeros((224, 224))
cam_a[80:140, 80:140] = 1.0   # center square

cam_b = np.zeros((224, 224))
cam_b[80:140, 80:140] = 1.0   # same region → should be 1.0

cam_c = np.zeros((224, 224))
cam_c[10:50, 10:50] = 1.0     # different region → should be 0.0

print(f"Same region: Jaccard={jaccard_similarity(cam_a, cam_b):.3f}, Spearman={spearman_agreement(cam_a, cam_b):.3f}")
print(f"Diff region: Jaccard={jaccard_similarity(cam_a, cam_c):.3f}, Spearman={spearman_agreement(cam_a, cam_c):.3f}")

---

## CELL 2: Compute Agreement Scores for Every Image

In [ ]:
import sys, os
from pathlib import Path

# ─── Self-contained setup ───
NOTEBOOK_DIR = Path(os.getcwd())
BACKEND_DIR = NOTEBOOK_DIR.parent.parent / "Skin_Lesion_Classification_backend"
ML_DIR = BACKEND_DIR / "ml"
if str(ML_DIR) not in sys.path:
    sys.path.insert(0, str(ML_DIR))

METADATA_PATH = ML_DIR / "data" / "processed" / "metadata_with_paths.csv"
MODEL_PATH = ML_DIR / "outputs" / "models" / "resnet50_best.pth"

if not METADATA_PATH.exists() or not MODEL_PATH.exists():
    print("❌ Prerequisites missing. Run 00_setup_and_sanity.ipynb first.")
    raise FileNotFoundError("Missing data or model.")

try:
    import torch
    import pandas as pd
    import numpy as np
    from PIL import Image
    from tqdm import tqdm

    from pytorch_grad_cam import GradCAM, GradCAMPlusPlus, EigenCAM
    from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
    from src.models.classifier import create_model, get_target_layer
    from src.data.dataset import get_transforms, create_splits
    import scipy.stats as stats

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    # Load model if not already
    try:
        model
    except NameError:
        model = create_model('resnet50', num_classes=2).to(device)
        model.load_state_dict(torch.load(MODEL_PATH, map_location=device))
        model.eval()
        target_layer = get_target_layer(model, 'resnet50')

    transform = get_transforms('test', 224)

    df = pd.read_csv(METADATA_PATH)
    _, _, test_df = create_splits(df)
    eval_df = test_df.sample(150, random_state=42)

    cam_methods = {
        'GradCAM':   GradCAM,
        'GradCAM++': GradCAMPlusPlus,
        'EigenCAM':  EigenCAM,
    }

    rq4_results = []

    for _, row in tqdm(eval_df.iterrows(), total=len(eval_df)):
        try:
            orig = np.array(Image.open(row['filepath']).convert('RGB').resize((224, 224)))
            input_t = transform(image=orig)['image'].unsqueeze(0).to(device)

            with torch.no_grad():
                out = model(input_t)
                probs = torch.softmax(out, dim=1)[0]
                pred = probs.argmax().item()
                confidence = probs[pred].item()
                correct = int(pred == int(row['label']))

            cams = {}
            for name, cam_class in cam_methods.items():
                with cam_class(model=model, target_layers=[target_layer]) as cam_gen:
                    cams[name] = cam_gen(input_tensor=input_t, targets=[ClassifierOutputTarget(pred)])[0]

            # Compute all pairwise agreements
            method_list = list(cams.keys())
            jaccard_scores = []
            spearman_scores = []
            for i in range(len(method_list)):
                for j in range(i+1, len(method_list)):
                    m1, m2 = method_list[i], method_list[j]
                    jaccard_scores.append(jaccard_similarity(cams[m1], cams[m2]))
                    spearman_scores.append(spearman_agreement(cams[m1], cams[m2]))

            rq4_results.append({
                'image_id':        row['image_id'],
                'dx':              row['dx'],
                'true_label':      row['label'],
                'predicted':       pred,
                'confidence':      confidence,
                'correct':         correct,
                'avg_jaccard':     np.mean(jaccard_scores),
                'avg_spearman':    np.mean(spearman_scores),
                'min_jaccard':     np.min(jaccard_scores),
            })

        except Exception as e:
            print(f"Error on {row['image_id']}: {e}")

    rq4_df = pd.DataFrame(rq4_results)
    OUTPUTS_DIR = NOTEBOOK_DIR / "outputs" / "metrics"
    OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)
    rq4_df.to_csv(OUTPUTS_DIR / "RQ4_agreement_results.csv", index=False)
    print(f"Completed: {len(rq4_df)} images")

except ImportError as e:
    print(f"❌ Backend module not found: {e}")
    print("The backend ML code is not implemented yet.")

---

## CELL 3: THE KEY ANALYSIS — Does Agreement Predict Correctness?

In [ ]:
# ─── Self-contained: check prerequisites ───
try:
    rq4_df
except NameError:
    print("❌ Results not computed. Run CELL 5 first.")
    raise NameError("Run CELL 5 first.")

# Group by correct/incorrect prediction
correct   = rq4_df[rq4_df['correct'] == 1]['avg_jaccard']
incorrect = rq4_df[rq4_df['correct'] == 0]['avg_jaccard']

print("=== Agreement by Prediction Correctness ===\n")
print(f"Correct predictions:   Jaccard mean={correct.mean():.4f}, std={correct.std():.4f}, n={len(correct)}")
print(f"Incorrect predictions: Jaccard mean={incorrect.mean():.4f}, std={incorrect.std():.4f}, n={len(incorrect)}")

# Mann-Whitney U test (non-parametric, for two independent groups)
stat, p = stats.mannwhitneyu(correct, incorrect, alternative='greater')
print(f"\nMann-Whitney U (correct > incorrect): stat={stat:.1f}, p={p:.4f}")
if p < 0.05:
    print("✅ Correct predictions have significantly higher inter-method agreement")
    print("→ SUPPORTS HYPOTHESIS: agreement predicts correctness")
else:
    print("❌ No significant difference — agreement doesn't reliably predict correctness")
    print("→ INTERESTING NEGATIVE RESULT: still worth reporting!")

# Correlation between confidence and agreement
r, p_corr = stats.pearsonr(rq4_df['confidence'], rq4_df['avg_jaccard'])
print(f"\nPearson r (confidence vs agreement): r={r:.4f}, p={p_corr:.4f}")
r_s, p_s = stats.spearmanr(rq4_df['confidence'], rq4_df['avg_jaccard'])
print(f"Spearman r (confidence vs agreement): r={r_s:.4f}, p={p_s:.4f}")

---

## CELL 4: Visualize Results

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Box plot: correct vs incorrect
data_plot = pd.DataFrame({
    'group': ['Correct']*len(correct) + ['Incorrect']*len(incorrect),
    'agreement': pd.concat([correct, incorrect]).values
})
sns.boxplot(data=data_plot, x='group', y='agreement', palette=['#2ecc71', '#e74c3c'], ax=axes[0])
axes[0].set_title('Inter-method Agreement\nvs Prediction Correctness', fontsize=11)
axes[0].set_ylabel('Avg Jaccard Similarity')

# Scatter: confidence vs agreement (colored by correct/incorrect)
colors = ['#2ecc71' if c else '#e74c3c' for c in rq4_df['correct']]
axes[1].scatter(rq4_df['confidence'], rq4_df['avg_jaccard'], c=colors, alpha=0.5, s=30)
axes[1].set_xlabel('Model Confidence')
axes[1].set_ylabel('Inter-method Agreement (Jaccard)')
axes[1].set_title(f'Confidence vs Agreement\nr={r:.3f}, p={p_corr:.4f}', fontsize=11)

# Distribution of agreement scores
rq4_df['correct_label'] = rq4_df['correct'].map({1: 'Correct', 0: 'Incorrect'})
sns.histplot(data=rq4_df, x='avg_jaccard', hue='correct_label', 
             bins=20, ax=axes[2], palette=['#2ecc71', '#e74c3c'])
axes[2].set_title('Distribution of Agreement Scores', fontsize=11)

plt.tight_layout()
plt.savefig('../outputs/figures/RQ4_agreement_analysis.png', dpi=150, bbox_inches='tight')
plt.show()